In [1]:
import re
from sqlalchemy import create_engine
import urllib
import pandas as pd 

In [2]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Shopify;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")

# Configurar cadena de conexión
params_com = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor
engine_com = create_engine(f"mssql+pyodbc:///?odbc_connect={params_com}")

In [3]:
query_data = """
SELECT DISTINCT c.Id AS Cliente,
       c.Nombre,
       c.Email AS Email,
       c.Telefono AS Celular,
       v.Fecha AS Fecha_Venta,
       v.Canal,
       SUM(v.Valor_Neto) AS Venta,
       SUM(Descuentos) AS Descuento
FROM Contacto_Clientes c
INNER JOIN Ventas_ShopifyMoviNova v ON c.id = v.id_cliente
WHERE v.fecha >= '2025-07-01'
    AND v.Fecha <= '2025-07-31'
    AND v.Codigo_Descuento LIKE '%CUMP%'
GROUP BY c.Id,
       c.Nombre,
       c.Email,
       c.Telefono,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
ventas_shopify = pd.read_sql(query_data, engine_data)

query_com = """
SELECT DISTINCT c.CLICodigo AS Cliente,
       c.CLINombres + ' ' + c.CLIApellidos AS Nombre,
       c.CLIEmailPrincipal AS Email,
       c.CLICelular AS Celular,
       v.Fecha AS Fecha_Venta,
       v.Canal,
       SUM(v.Venta_Neta) AS Venta,
       SUM(Descuento) AS Descuento
FROM Contacto_Clientes c
INNER JOIN Ventas_Comerssia v ON c.CLICodigo = v.Cliente
WHERE v.Fecha >= '2025-07-01'
    AND v.Fecha <= '2025-07-31'
    AND v.ICPDescripcion LIKE '%CUMP%'
GROUP BY  c.CLICodigo,
       c.CLINombres + ' ' + c.CLIApellidos,
       c.CLIEmailPrincipal,
       c.CLICelular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
ventas_comerssia = pd.read_sql(query_com, engine_com)

df_ventas = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)
df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,Descuento
0,C1000192914,ANA SOFIA RIVERA GOMEZ,riveragomezanasofia@gmail.com,3168149182,2025-07-22 00:00:00,Retail,123529.41,147000.0
1,C1000547706,VERONICA MELO,veronicamelovs21@gmail.com,3104645444,2025-07-24 00:00:00,Retail,439285.72,174250.0
2,C1000654984,ANDREA ZAPATA GOMEZ,azapatagomez6@gmail.com,3012196755,2025-07-12 00:00:00,Retail,64285.71,25500.0
3,C1001367138,ALICIA VELEZ VELASQUEZ,negado@provenzal.net,3206175031,2025-07-02 00:00:00,Retail,132352.94,52500.0
4,C1001848669,VANNESA VELEZ LOPEZ,vannesavelez@hotmail.com,3015809698,2025-07-17 00:00:00,Retail,730672.26,869500.0


In [4]:
ruta_excel = 'Cumpleaños JULIO.xlsx'
df_plan_a = pd.read_excel(ruta_excel, sheet_name='PLAN A')
df_plan_b = pd.read_excel(ruta_excel, sheet_name='PLAN B ')

df_plan_a['Cliente'] = df_plan_a['Cliente'].astype(str)
df_plan_b['Cliente'] = df_plan_b['Cliente'].astype(str)

# Función para limpiar espacios y quitar la "C" inicial si la tiene
def limpiar_codigo(codigo):
    return re.sub(r'^C', '', codigo.strip())

# Aplicar la función a cada DataFrame
df_ventas['Cliente'] = df_ventas['Cliente'].apply(limpiar_codigo)
df_plan_a['Cliente'] = df_plan_a['Cliente'].apply(limpiar_codigo)
df_plan_b['Cliente'] = df_plan_b['Cliente'].apply(limpiar_codigo)


In [5]:
ventas_plan_a = df_ventas[df_ventas['Cliente'].isin(df_plan_a['Cliente'])]
total_clientes_a = ventas_plan_a['Cliente'].nunique()
total_ventas_a = ventas_plan_a['Venta'].sum()
total_descuento_a = ventas_plan_a['Descuento'].sum()

# 4. Filtrar ventas por clientes de Plan B
ventas_plan_b = df_ventas[df_ventas['Cliente'].isin(df_plan_b['Cliente'])]
total_clientes_b = ventas_plan_b['Cliente'].nunique()
total_ventas_b = ventas_plan_b['Venta'].sum()
total_descuento_b = ventas_plan_b['Descuento'].sum()

In [6]:
# 5. Mostrar resultados
print("Plan A:")
print(f"Clientes que compraron: {total_clientes_a}")
print(f"Total ventas: ${total_ventas_a:,.0f}")
print(f"Total descuento: ${total_descuento_a:,.0f}")

print("\nPlan B:")
print(f"Clientes que compraron: {total_clientes_b}")
print(f"Total ventas: ${total_ventas_b:,.0f}")
print(f"Total descuento: ${total_descuento_b:,.0f}")

Plan A:
Clientes que compraron: 303
Total ventas: $118,344,127
Total descuento: $46,883,500

Plan B:
Clientes que compraron: 249
Total ventas: $93,682,569
Total descuento: $37,250,750
